In [ ]:
import os

try:
    import google.colab
    IS_COLAB = True
    print("Google Colab")
except ImportError:
    IS_COLAB = False
    print("로컬 Jupyter")

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("import 완료")
print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

로컬 Jupyter
import 완료
API 키 OK


In [ ]:
from langchain_openai import ChatOpenAI # LangChain에서 OpenAI 채팅 모델을 사용하기 위한 클래스
from langchain_core.prompts import ChatPromptTemplate   # LLM에게 전달할 프롬프트의 형식을 만들기 위한 클래스
from datetime import datetime

"""LLM 설정 코드"""
llm = ChatOpenAI(
    model="gpt-4o-mini",    # 사용할 OpenAI 모델
    temperature=0.3         # 답변의 다양성 정도(낮을수록 비교적 일관적인 답변)
)


# 페르소나 설정 코드
# - Python의 딕셔너리를 이용해서 여러 개의 페르소나를 저장
# - key: 페르소나 이름
# - value: 실제로 LLM에게 전달할 역할 및 행동 지시

PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}


# 프롬프트 템플릿 설정
# - LLM에게 전달할 메시지의 구조를 정의
# - system: AI 역할이나 행동 방식을 지정
# - human: 사용자가 실제로 입력한 질문
# - {persona}, {question}은 아직 실제 값이 아니라
# - 나중에 invoke()를 실행할 때 값이 들어갈 자리

prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),
    ("human", "{question}")
])

# 대화 누적 리스트
log = []  

"""프롬프트와 LLM을 연결하여 체인 생성"""
# | 는 LangChain의 LCEL 파이프 연산자
# 처리 순서:
# 입력값
#   ↓
# prompt
#   ↓
# llm
#   ↓
# AI 답변
# 즉, prompt에서 메시지를 완성한 다음 그 결과를 llm에게 전달하도록 연결
chain = prompt | llm

"""챗봇 핵심 함수"""
# persona_name : 사용할 페르소나의 이름
# question     : 사용자가 입력한 질문
# : str  → 문자열을 받는다는 타입 힌트
# -> str → 최종적으로 문자열을 반환한다는 타입 힌트
def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]

    result = chain.invoke({
        "persona": system_message,
        "question": question
    })

    return result.content

def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)    # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ")
        user_input = user_input.strip()
        
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

    log.append(f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {user_input} → {answer}")

    # 종료 시:
    with open("chat_log.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(log))

In [4]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 저는 데이터 분석과 관련된 질문에 답변하고 정보를 제공하는 AI입니다. 데이터 분석, 통계, 머신러닝 등 다양한 주제에 대해 도움을 드릴 수 있습니다. 필요한 정보나 질문이 있으시면 말씀해 주세요!

[분석가] 데이터 분석은 수집된 데이터를 정리, 처리, 해석하여 유용한 정보를 도출하고 의사 결정을 지원하는 과정입니다. 이 과정은 일반적으로 다음과 같은 단계로 이루어집니다:

1. **데이터 수집**: 다양한 출처에서 데이터를 수집합니다.
2. **데이터 정제**: 결측치, 이상치 등을 처리하여 데이터의 품질을 높입니다.
3. **데이터 탐색**: 통계적 기법이나 시각화를 통해 데이터의 패턴과 트렌드를 파악합니다.
4. **모델링**: 예측 모델이나 분류 모델을 구축하여 데이터를 분석합니다.
5. **결과 해석**: 분석 결과를 기반으로 인사이트를 도출하고, 이를 의사 결정에 활용합니다.

예를 들어, 기업이 고객 데이터를 분석하여 구매 패턴을 이해하고 마케팅 전략을 최적화하는 경우가 이에 해당합니다. 데이터 분석은 비즈니스, 의료, 금융 등 다양한 분야에서 중요한 역할을 합니다.

👋 종료합니다.
